# 🌿 Smart Recycling AI — Training on Google Colab

This notebook trains **EfficientNet-B2** for waste classification.

**Steps:**
1. Install dependencies
2. Upload your project files (config + model folder)
3. Download & prepare TrashNet dataset from Kaggle
4. Train the model (~40 mins on T4 GPU)
5. Download `best.pt` to your Mac

> ⚡ Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**

## Step 1 — Enable GPU
Go to **Runtime → Change runtime type → Hardware Accelerator → T4 GPU → Save**

Then run the cell below to confirm GPU is available:

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q timm albumentations torch torchvision scikit-learn tqdm pyyaml matplotlib seaborn kaggle

## Step 3 — Upload Your Project Files

Run the cell below and upload the **zip file** of your project (instructions below the cell).

In [ ]:
from google.colab import files
import zipfile, os

print('📁 Upload your project zip file (waste_project.zip)...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/waste_project')

os.chdir('/content/waste_project')
print('✅ Project extracted. Current directory:', os.getcwd())
!ls

## Step 4 — Setup Kaggle API & Download TrashNet

In [ ]:
from google.colab import files
import os

print('📁 Upload your kaggle.json API token...')
uploaded = files.upload()  # upload kagwha gle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle API configured')

In [ ]:
# Download TrashNet dataset
!mkdir -p data/raw
!kaggle datasets download -d asdasdasasdas/garbage-classification -p data/raw --unzip
!ls data/raw/

In [ ]:
# Fix nested folder structure if needed
import os, shutil
from pathlib import Path

nested = Path('data/raw/Garbage classification/Garbage classification')
if nested.exists():
    for item in nested.iterdir():
        shutil.move(str(item), 'data/raw/')
    shutil.rmtree('data/raw/Garbage classification')
    print('✅ Fixed nested folder structure')

!ls data/raw/

## Step 5 — Prepare Dataset

In [ ]:
import sys
sys.path.insert(0, 'model/training')
sys.path.insert(0, 'model/inference')

!python scripts/prepare_dataset.py

## Step 6 — Train EfficientNet-B2 🚀

This will take ~40 minutes on T4 GPU (vs 5+ hours on your Mac CPU).

In [ ]:
import yaml

# Update config for GPU training
with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['training']['batch_size'] = 32
cfg['training']['epochs'] = 30
cfg['training']['patience'] = 7
cfg['training']['num_workers'] = 2
cfg['training']['device'] = 'cuda'

with open('config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ Config updated for GPU training')
print(f"  batch_size : {cfg['training']['batch_size']}")
print(f"  epochs     : {cfg['training']['epochs']}")
print(f"  device     : {cfg['training']['device']}")

In [ ]:
!python model/training/train.py

## Step 7 — Download the Trained Model ⬇️

In [ ]:
from google.colab import files
import os

best_pt = 'model/weights/best.pt'
if os.path.exists(best_pt):
    size_mb = os.path.getsize(best_pt) / 1e6
    print(f'✅ best.pt ready — {size_mb:.1f} MB')
    files.download(best_pt)
    print('⬇️ Download started!')
else:
    print('❌ best.pt not found. Did training complete?')

## ✅ Done!

After downloading `best.pt`:
1. Move it to `model/weights/best.pt` on your Mac
2. Run `python main.py` to start the API
3. Your `/classify` endpoint is now live!